# **Gamma Earth S2DR4 - QGIS API SERVER**
## Backend Khusus untuk Plugin QGIS S2DR4 Connector

**[DIBUAT OLEH CAT SPATIAL SPECIALIST]**
Notebook ini didesain **KHUSUS** untuk menjadi *API Server* di balik layar untuk Plugin QGIS. Tidak ada titik *dummy* yang akan diproses.
Cukup tekan **Runtime -> Run All**, dan *copy link* Ngrok di sel paling bawah ke dalam QGIS-mu!

# **1. MOUNT GOOGLE DRIVE & SETUP FOLDER**

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

drive_output_dir = '/content/drive/MyDrive/S2DR4_Output'
os.makedirs(drive_output_dir, exist_ok=True)
print(f"[INFO] Direktori output di Google Drive: {drive_output_dir}")

output_dir = '/content/output'
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)
print(f"[INFO] Direktori output lokal (sementara): {output_dir}")


# **2. SETUP VIRTUAL ENVIRONMENT PYTHON 3.12 & KOMPILASI GDAL MURNI**

In [ ]:
%%bash
set -e
echo "[+] Menginstall Python 3.12, build tools, dan GDAL headers..."
apt-get update -yqq
apt-get install -yqq python3.12 python3.12-dev python3.12-venv build-essential libgdal-dev gdal-bin

echo "[+] Membuat VENV MURNI Python 3.12..."
rm -rf /content/s2dr4_env
/usr/bin/python3.12 -m venv /content/s2dr4_env --without-pip

echo "[+] Menyuntikkan PIP secara manual..."
wget -q https://bootstrap.pypa.io/get-pip.py
/content/s2dr4_env/bin/python get-pip.py --quiet
rm get-pip.py

echo "[+] Install numpy 1.x dan build tools..."
/content/s2dr4_env/bin/pip install --quiet "numpy<2" setuptools wheel Cython

echo "[+] Mengompilasi GDAL Python bindings (proses kompilasi native)..."
GDAL_VERSION=$(/usr/bin/gdal-config --version)
echo "    Versi GDAL terdeteksi: $GDAL_VERSION"
export CPLUS_INCLUDE_PATH=/usr/include/gdal
export C_INCLUDE_PATH=/usr/include/gdal
/content/s2dr4_env/bin/pip install --quiet --no-build-isolation "GDAL==$GDAL_VERSION"
echo "[DONE] GDAL $GDAL_VERSION berhasil dikompilasi!"


# **3. INSTALL DEPENDENSI SPASIAL DAN S2DR4**

In [ ]:
%%bash
set -e
echo "[+] Menginstall dependensi yang membutuhkan numpy 1.x..."
/content/s2dr4_env/bin/pip install --quiet \
    "scipy<1.12" "tifffile<2023.1" \
    xarray rioxarray rasterio earthengine-api geemap \
    tqdm torch torchvision pandas scikit-learn matplotlib \
    pyparsing oauthlib requests-oauthlib google-auth-oauthlib gspread arosics \
    mgrs h3 boto3 einops fiona geopy google-cloud icecream pystac-client stackstac utm xmltodict

# ==================================================================================
# PATCH CAT SPATIAL SPECIALIST: Menginstall imagecodecs murni untuk Kompresi LZW
# Diinstal menggunakan '--no-deps' untuk memblokir Numpy agar tidak ter-upgrade
# ==================================================================================
echo "[+] Menginstall imagecodecs murni (kompresi LZW)..."
/content/s2dr4_env/bin/pip install --quiet --no-deps imagecodecs

echo "[+] Menginstall S2DR4 (--no-deps untuk memblokir override package)..."
/content/s2dr4_env/bin/pip install --quiet --no-deps \
    https://storage.googleapis.com/0x7ff601307fa5/s2dr4-20260205.1-cp312-cp312-linux_x86_64.whl

echo ""
echo "[+] ======= UJI COBA S2DR4 FINAL ======="
/content/s2dr4_env/bin/python -c "
import numpy; print(f'NumPy: {numpy.__version__}')
from osgeo import gdal; print(f'GDAL:  {gdal.__version__}')
import imagecodecs; print('[OK] IMAGECODECS BERHASIL DISUNTIKKAN')
import s2dr4.inferutils; print('[OK] S2DR4 BERHASIL TERINSTAL SEMPURNA!')
"


# **4. COLAB API SERVER (Ngrok + FastAPI)**
Sel ini akan terus berjalan (*running*) untuk mendengarkan instruksi dari QGIS-mu.

In [ ]:
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio python-multipart

import os, nest_asyncio, asyncio
from pyngrok import ngrok
from google.colab import userdata
from uvicorn import Config, Server
from fastapi import FastAPI
from fastapi.responses import FileResponse
from pydantic import BaseModel
import subprocess

# 1. Setup Ngrok Auth (Ambil dari Colab Secrets)
try:
    NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    print("\n[ERROR] NGROK_AUTHTOKEN tidak ditemukan di Colab Secrets!")
    print("Silakan tambahkan NGROK_AUTHTOKEN di menu Secrets (ikon kunci di kiri).\n")
    raise e

# 2. Mulai Ngrok Tunnel
ngrok_tunnel = ngrok.connect(8000)
print(f"\n[OK] API Server Berjalan di: {ngrok_tunnel.public_url}\n")
print("COPY URL DI ATAS DAN PASTE KE DALAM PLUGIN QGIS ANDA!\n")

# 3. Konfigurasi FastAPI
app = FastAPI(title="S2DR4 API Server")
nest_asyncio.apply()

class InferRequest(BaseModel):
    lon: float
    lat: float
    date: str

@app.post("/infer")
async def run_inference(request: InferRequest):
    print(f"\n[API REQUEST] Memproses koordinat: ({request.lon}, {request.lat}) Tanggal: {request.date}")
    
    # Create python code for dynamic generation
    real_python_code = [
        "import s2dr4.inferutils\n",
        "import os, time, shutil\n",
        "from pathlib import Path\n\n",
        "# Hapus output lama agar tidak terkirim ulang\n",
        "if os.path.exists('/content/output'):\n",
        "    shutil.rmtree('/content/output', ignore_errors=True)\n\n",
        "# Eksekusi S2DR4\n",
        f"s2dr4.inferutils.test(lonlat=({request.lon}, {request.lat}), date='{request.date}')\n",
        "time.sleep(5)\n",
        "current_files = list(Path('/content/output').rglob('*_MS.tif'))\n",
        "if current_files:\n",
        "    print(str(current_files[0]))\n"
    ]
    
    with open('/content/api_infer.py', 'w') as f:
        f.writelines(real_python_code)
    
    try:
        result = subprocess.run(['/content/s2dr4_env/bin/python', '/content/api_infer.py'], capture_output=True, text=True)
        output_lines = result.stdout.strip().split('\n')
        tif_path = None
        for line in reversed(output_lines):
            if line.endswith('_MS.tif') and os.path.exists(line):
                tif_path = line
                break
                
        if tif_path:
            print(f"[API SUCCESS] Mengirim file: {os.path.basename(tif_path)}")
            return FileResponse(path=tif_path, filename=os.path.basename(tif_path), media_type="image/tiff")
        else:
            return {"error": "File TIF gagal digenerate.", "log": result.stdout[-500:], "stderr": result.stderr[-500:]}
    except Exception as e:
        return {"error": str(e)}

# 4. Jalankan Server di Asyncio Loop bawaan Jupyter
config = Config(app, host="127.0.0.1", port=8000)
server = Server(config=config)
await server.serve()
